## 确认环境是否正常

In [1]:
import torch
import torchvision

In [9]:
torch.__version__

'2.5.1'

In [10]:
torchvision.__version__

'0.20.1'

In [4]:
torch.cuda.is_available()

True

In [6]:
torch.version.cuda

'12.4'

In [7]:
torch.cuda.get_device_name(0)

'NVIDIA GeForce GTX 1050'

## MNIST训练脚本测试

In [12]:
# Cell 1: 导入库
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.5.1
CUDA available: True


In [19]:
# Cell 2: 定义模型（一个简单的卷积神经网络）
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output

In [20]:
# Cell 3: 加载数据（首次运行会自动下载）
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = DataLoader(testset, batch_size=1000, shuffle=False)

In [21]:
# Cell 4: 初始化模型、损失函数、优化器
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Using device: {device}")

Using device: cuda


In [22]:
# Cell 5: 训练模型（只跑2个epoch，很快）
epochs = 2
for epoch in range(epochs):
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        if i % 100 == 99:
            print(f'Epoch {epoch+1}, Batch {i+1}, Loss: {running_loss/100:.4f}')
            running_loss = 0.0
    
    print(f'Epoch {epoch+1} finished')

print("Training complete!")

Epoch 1, Batch 100, Loss: 0.6878
Epoch 1, Batch 200, Loss: 0.2365
Epoch 1, Batch 300, Loss: 0.1880
Epoch 1, Batch 400, Loss: 0.1597
Epoch 1, Batch 500, Loss: 0.1361
Epoch 1, Batch 600, Loss: 0.1150
Epoch 1, Batch 700, Loss: 0.1171
Epoch 1, Batch 800, Loss: 0.1078
Epoch 1, Batch 900, Loss: 0.1116
Epoch 1 finished
Epoch 2, Batch 100, Loss: 0.0807
Epoch 2, Batch 200, Loss: 0.0774
Epoch 2, Batch 300, Loss: 0.0744
Epoch 2, Batch 400, Loss: 0.0888
Epoch 2, Batch 500, Loss: 0.0849
Epoch 2, Batch 600, Loss: 0.0829
Epoch 2, Batch 700, Loss: 0.0802
Epoch 2, Batch 800, Loss: 0.0763
Epoch 2, Batch 900, Loss: 0.0905
Epoch 2 finished
Training complete!


In [23]:
# Cell 6: 保存模型
torch.save(model.state_dict(), 'mnist_model.pth')
print("Model saved as mnist_model.pth")

Model saved as mnist_model.pth


In [24]:
# Cell 7: 快速验证（测试一张图片）
model.eval()
test_images, test_labels = next(iter(testloader))
test_images, test_labels = test_images.to(device), test_labels.to(device)

with torch.no_grad():
    outputs = model(test_images)
    _, predicted = torch.max(outputs, 1)

print(f"First test image - Predicted: {predicted[0].item()}, Actual: {test_labels[0].item()}")

First test image - Predicted: 7, Actual: 7


## 2026-04-19 周一完成
- PyTorch环境已确认（CUDA可用）
- MNIST训练脚本跑通
- 模型已保存为 mnist_model.pth
- 验证推理成功